In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import r2_score
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')

TRAIN_PATH = 'train.csv'
TEST_PATH  = 'test.csv'
OUT_PATH   = 'gridlock_optimized_submission.csv'
SEED = 42
N_FOLDS = 5

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
print('Train:', train.shape, '| Test:', test.shape)

Train: (77299, 11) | Test: (41778, 10)


## Feature Engineering

In [2]:
def parse_time(df):
    """Parse HH:MM timestamp into rich temporal features."""
    parts = df['timestamp'].astype(str).str.split(':', expand=True)
    hour   = pd.to_numeric(parts[0], errors='coerce').fillna(0).astype(int)
    minute = pd.to_numeric(parts[1], errors='coerce').fillna(0).astype(int)
    mod    = hour * 60 + minute

    df['hour']          = hour
    df['minute']        = minute
    df['minute_of_day'] = mod
    df['minute_15']     = minute // 15
    df['minute_30']     = minute // 30

    # Cyclic encoding — better than raw hour for trees too
    df['sin_hour']  = np.sin(2 * np.pi * hour / 24)
    df['cos_hour']  = np.cos(2 * np.pi * hour / 24)
    df['sin_time']  = np.sin(2 * np.pi * mod / 1440)
    df['cos_time']  = np.cos(2 * np.pi * mod / 1440)

    # Time-of-day buckets
    df['is_rush_am']  = hour.between(7, 10).astype(int)
    df['is_rush_pm']  = hour.between(17, 20).astype(int)
    df['is_rush']     = (df['is_rush_am'] | df['is_rush_pm']).astype(int)
    df['is_night']    = ((hour <= 5) | (hour >= 22)).astype(int)
    df['is_morning']  = hour.between(6, 11).astype(int)
    df['is_afternoon']= hour.between(12, 16).astype(int)
    df['is_evening']  = hour.between(16, 21).astype(int)

    return df


def add_geo_features(df):
    """Geohash prefix features for spatial hierarchy."""
    geo = df['geohash'].astype(str)
    df['geo4'] = geo.str[:4]
    df['geo5'] = geo.str[:5]
    df['geo6'] = geo.str[:6]   # == full geohash for 6-char hashes
    return df


def add_road_features(df):
    """Road-level interaction features."""
    df['lanes_x_landmark'] = (
        df['NumberofLanes'] * (df['Landmarks'] == 'Yes').astype(int)
    )
    df['is_highway'] = (df['RoadType'] == 'Highway').astype(int)
    df['is_residential'] = (df['RoadType'] == 'Residential').astype(int)
    df['large_highway'] = (
        (df['RoadType'] == 'Highway') &
        (df['LargeVehicles'] == 'Allowed')
    ).astype(int)
    return df


train = parse_time(train)
test  = parse_time(test)
train = add_geo_features(train)
test  = add_geo_features(test)
train = add_road_features(train)
test  = add_road_features(test)

print('Features after base engineering:', train.shape[1])

Features after base engineering: 34


## OOF Target Encoding (the most important upgrade)

**Why**: `geohash × timestamp` mean demand correlates 0.987 with the target.
Without OOF (out-of-fold) protection, we'd overfit. OOF encoding computes
the mean from the *other* folds, preventing leakage.

We encode: geohash, timestamp, geo×timestamp, geo4×hour, geo5×hour, day×hour

In [3]:
def oof_target_encode(train_df, test_df, group_cols, target_col, n_folds=5, seed=42, suffix=None):
    """
    Out-of-fold target encoding.
    Returns encoded train column and test column (test uses full-train mean).
    """
    key = '__'.join(group_cols) if suffix is None else suffix
    col_name = f'te_{key}'

    # Create group key
    if len(group_cols) == 1:
        train_key = train_df[group_cols[0]].astype(str)
        test_key  = test_df[group_cols[0]].astype(str)
    else:
        train_key = train_df[group_cols].astype(str).agg('_'.join, axis=1)
        test_key  = test_df[group_cols].astype(str).agg('_'.join, axis=1)

    y = train_df[target_col].values
    oof_enc = np.zeros(len(train_df))

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    global_mean = np.mean(y)

    for tr_idx, va_idx in kf.split(train_df):
        fold_map = (
            pd.Series(y[tr_idx], index=train_key.iloc[tr_idx])
            .groupby(level=0).mean()
        )
        oof_enc[va_idx] = train_key.iloc[va_idx].map(fold_map).fillna(global_mean).values

    # Test encoding: use full-train mean
    full_map = pd.Series(y, index=train_key).groupby(level=0).mean()
    test_enc = test_key.map(full_map).fillna(global_mean).values

    return oof_enc, test_enc, col_name


TARGET = 'demand'

# Define encoding groups (most powerful first)
encode_groups = [
    (['geohash', 'timestamp'],          'geo_ts'),       # 0.987 corr!
    (['geohash', 'hour'],               'geo_hour'),
    (['geohash'],                        'geo'),
    (['geo5', 'hour'],                   'geo5_hour'),
    (['geo4', 'hour'],                   'geo4_hour'),
    (['geohash', 'minute_15'],           'geo_min15'),
    (['timestamp'],                      'ts'),
    (['day', 'hour'],                    'day_hour'),
    (['geohash', 'day'],                 'geo_day'),
    (['RoadType', 'hour'],               'road_hour'),
    (['geohash', 'day', 'hour'],         'geo_day_hour'),
]

te_train_cols = {}
te_test_cols  = {}

for cols, suffix in encode_groups:
    tr_enc, te_enc, col_name = oof_target_encode(
        train, test, cols, TARGET, n_folds=N_FOLDS, seed=SEED, suffix=suffix
    )
    te_train_cols[col_name] = tr_enc
    te_test_cols[col_name]  = te_enc
    print(f'  {col_name}: train corr = {np.corrcoef(tr_enc, train[TARGET].values)[0,1]:.4f}')

# Append to dataframes
for col_name, vals in te_train_cols.items():
    train[col_name] = vals
for col_name, vals in te_test_cols.items():
    test[col_name] = vals

print('\nFeatures after target encoding:', train.shape[1])

  te_geo_ts: train corr = 0.2198
  te_geo_hour: train corr = 0.9465
  te_geo: train corr = 0.8287
  te_geo5_hour: train corr = 0.4242
  te_geo4_hour: train corr = 0.2081
  te_geo_min15: train corr = 0.8149
  te_ts: train corr = 0.1240
  te_day_hour: train corr = 0.1422
  te_geo_day: train corr = 0.8439
  te_road_hour: train corr = 0.8674
  te_geo_day_hour: train corr = 0.9543

Features after target encoding: 45


## Statistical Aggregation Features
Per-geohash: std, min, max, p25, p75 of demand (from train only — no leakage on test)


In [4]:
def add_stat_features(train_df, test_df):
    """Add geohash-level demand statistics (computed only from train)."""
    agg = (
        train_df.groupby('geohash')['demand']
        .agg(
            geo_std='std',
            geo_min='min',
            geo_max='max',
            geo_p25=lambda x: x.quantile(0.25),
            geo_p75=lambda x: x.quantile(0.75),
            geo_median='median',
            geo_skew='skew',
        )
        .reset_index()
    )

    train_df = train_df.merge(agg, on='geohash', how='left')
    test_df  = test_df.merge(agg, on='geohash', how='left')

    # Geohash frequency (proxy for urban density)
    freq = train_df['geohash'].value_counts().rename('geo_freq')
    train_df['geo_freq'] = train_df['geohash'].map(freq)
    test_df['geo_freq']  = test_df['geohash'].map(freq).fillna(freq.median())

    # Hour-level demand stats
    hour_agg = (
        train_df.groupby('hour')['demand']
        .agg(hour_mean='mean', hour_std='std')
        .reset_index()
    )
    train_df = train_df.merge(hour_agg, on='hour', how='left')
    test_df  = test_df.merge(hour_agg, on='hour', how='left')

    return train_df, test_df

train, test = add_stat_features(train, test)
print('Features after stat aggregation:', train.shape[1])

Features after stat aggregation: 55


## Ordinal Encoding + Train/Test Prep

In [5]:
DROP_COLS = ['demand', 'Index', 'geohash', 'timestamp']  # raw strings no longer needed

def encode_and_split(train_df, test_df):
    y = train_df[TARGET].values

    X = train_df.drop(columns=[c for c in DROP_COLS if c in train_df.columns])
    T = test_df.drop(columns=[c for c in DROP_COLS if c in test_df.columns and c != 'Index'])

    # Align columns
    common = [c for c in X.columns if c in T.columns]
    X = X[common]
    T = T[common]

    all_df = pd.concat([X, T], ignore_index=True)
    cat_cols = all_df.select_dtypes(include='object').columns.tolist()
    num_cols = [c for c in all_df.columns if c not in cat_cols]

    for c in cat_cols:
        all_df[c] = all_df[c].fillna('NA').astype(str)
    for c in num_cols:
        all_df[c] = pd.to_numeric(all_df[c], errors='coerce')
        all_df[c] = all_df[c].fillna(all_df[c].median())

    if cat_cols:
        enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        all_df[cat_cols] = enc.fit_transform(all_df[cat_cols])

    X_enc = all_df.iloc[:len(train_df)].astype('float32').reset_index(drop=True)
    T_enc = all_df.iloc[len(train_df):].astype('float32').reset_index(drop=True)

    return X_enc, y, T_enc

X, y, T = encode_and_split(train, test)
print('X shape:', X.shape, '| T shape:', T.shape)
print('Feature list sample:', X.columns.tolist()[:15])

X shape: (77299, 51) | T shape: (41778, 51)
Feature list sample: ['day', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather', 'hour', 'minute', 'minute_of_day', 'minute_15', 'minute_30', 'sin_hour', 'cos_hour', 'sin_time']


## Optional: log1p-transform target
Demand is right-skewed. Training on `log1p(demand)` and exponentiating predictions
often improves R² by 0.5–2 points on skewed targets.

In [6]:
USE_LOG_TARGET = True  # Set False to compare

if USE_LOG_TARGET:
    y_train = np.log1p(y)
    print('Using log1p target. Max raw demand:', y.max())
else:
    y_train = y

Using log1p target. Max raw demand: 1.0


## Model Definitions (tuned hyperparameters)

In [7]:
models = [
    ('lgbm', LGBMRegressor(
        n_estimators=3000,
        learning_rate=0.02,
        num_leaves=127,        # increased from 64
        max_depth=10,
        min_child_samples=20,
        subsample=0.80,
        colsample_bytree=0.80,
        reg_alpha=0.05,
        reg_lambda=0.3,
        random_state=SEED,
        n_jobs=-1,
        verbosity=-1,
    )),
    ('xgb', XGBRegressor(
        n_estimators=2000,
        learning_rate=0.02,
        max_depth=9,
        min_child_weight=5,
        subsample=0.80,
        colsample_bytree=0.80,
        gamma=0.1,
        reg_alpha=0.05,
        reg_lambda=0.3,
        objective='reg:squarederror',
        tree_method='hist',
        random_state=SEED+1,
        n_jobs=-1,
    )),
    ('cat', CatBoostRegressor(
        iterations=2000,
        learning_rate=0.025,
        depth=9,
        l2_leaf_reg=3,
        loss_function='RMSE',
        random_seed=SEED+2,
        verbose=False,
        allow_writing_files=False,
    ))
]

## Training — OOF + Stacking

In [8]:
from sklearn.linear_model import Ridge

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_matrix  = np.zeros((len(X), len(models)))   # for stacking
test_preds  = []

print('Training ensemble...')
for m_idx, (name, model) in enumerate(models):
    oof  = np.zeros(len(X))
    pred = np.zeros(len(T))

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X), 1):
        model.fit(X.iloc[tr_idx], y_train[tr_idx])
        oof[va_idx] = model.predict(X.iloc[va_idx])
        pred += model.predict(T) / N_FOLDS

    # Back-transform if log target was used
    if USE_LOG_TARGET:
        oof_raw  = np.expm1(oof)
        pred_raw = np.expm1(pred)
    else:
        oof_raw  = oof
        pred_raw = pred

    score = r2_score(y, np.clip(oof_raw, 0, 1))
    print(f'  {name} OOF R²: {score:.6f}')

    oof_matrix[:, m_idx] = oof_raw
    test_preds.append(pred_raw)

# ── Stacking meta-learner (Ridge) ──────────────────────────────────────────
meta = Ridge(alpha=1.0)
meta.fit(oof_matrix, y)
meta_weights = meta.coef_ / meta.coef_.sum()
print('\nMeta-learner weights:', dict(zip([n for n,_ in models], meta_weights.round(3))))

blend = sum(w * p for w, p in zip(meta_weights, test_preds))

# OOF stacking score
oof_blend = oof_matrix @ meta_weights
stack_score = r2_score(y, np.clip(oof_blend, 0, 1))
print(f'Stacked OOF R²: {stack_score:.6f}')

blend = np.clip(blend, 0, 1)

Training ensemble...
  lgbm OOF R²: 0.962454
  xgb OOF R²: 0.957729
  cat OOF R²: 0.963006

Meta-learner weights: {'lgbm': 0.417, 'xgb': 0.079, 'cat': 0.504}
Stacked OOF R²: 0.963745


## Generate Submission

In [9]:
sub = pd.DataFrame({'Index': test['Index'], 'demand': blend})
sub.to_csv(OUT_PATH, index=False)
print('Saved:', OUT_PATH)
print(sub.shape)
print(sub.head())

Saved: gridlock_optimized_submission.csv
(41778, 2)
   Index    demand
0      0  0.059562
1      1  0.026544
2      2  0.038669
3      3  0.039082
4      4  0.064354


## BONUS: Optuna Hyperparameter Tuning (run separately, ~15 min)
Paste this into a separate cell and run after the main submission is done.
Can squeeze another 0.01–0.02 R².